# 03 M9

Scores every site-day with M9 from the `pynrpf` package (label-free), fits the two calibration numbers per fold on the fold's calibration stations, and decides every held-out site-day at the control c.

Abbreviations used here: **RPF** is reverse power flow, the condition where a distribution substation exports power because rooftop solar exceeds local demand; a *wrong RPF sign* is a meter recording that stores the export as an import. **M7** is the deterministic threshold rule, **M8** the two-stage XGBoost classifier and **M9** the counterfactual bridge method of the `pynrpf` package. **MW** and **MWh** are megawatts and megawatt-hours; one interval is 15 minutes, and a *slot* counts intervals from midnight (slot 24 is 06:00).

**Inputs.** `results/01_data_folds/` and the datasets; the `pynrpf` package.

**Outputs.** `results/03_m9/`: `scores_alpha.parquet` and `scores_beta.parquet` (the label-free evidence and windows of every site-day), `calibration_fits.csv` (the two numbers per fold and the raw evidence thresholds they imply), `site_days_m9.parquet` (probability and outcome per held-out site-day), `intervals_m9.parquet`; `results/manifests/03_m9.json`.

**Runtime.** About two minutes.

**Steps.**

1. Setup.
2. Score, calibrate per fold, decide.
3. Read the calibration fits and the outcome counts.

## 1. Setup

Locate the article folder, import the paper code and load the configuration. Loading the configuration verifies the SHA-256 of every dataset, so a wrong or edited data file stops the run here. `CONFIG` is the one knob: point it at another YAML to run a variant into another folder.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Image, Markdown, display


def article_root() -> Path:
    """publication/2_journal_article, found from this folder, JupyterLab's root or the repository root."""
    for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (candidate / "paper" / "stages.py").exists():
            return candidate
        nested = candidate / "publication" / "2_journal_article"
        if (nested / "paper" / "stages.py").exists():
            return nested
    raise FileNotFoundError("Could not locate publication/2_journal_article.")


ARTICLE = article_root()
sys.path.insert(0, str(ARTICLE))
from paper import config, results, stages  # noqa: E402

CONFIG = ARTICLE / "config" / "evaluation.yaml"   # point this at another configuration to run a variant
SETTINGS = config.load(CONFIG)                     # verifies the dataset hashes before anything runs
RESULTS = SETTINGS.output_root()
print("results folder:", RESULTS.relative_to(ARTICLE))

## 2. Score, calibrate, decide

Every site-day is scored once: the 1,176 candidate windows, the straight-bridge test, the evidence of the best window. The evidence floor is the smallest non-zero overnight demand step of each dataset, computed label-free. For each fold the two calibration numbers are fitted by logistic regression on the reviewed sure days of the fold's calibration stations, then applied to the held-out station at c = 0.7.

In [ ]:
out = stages.m9(SETTINGS)

## 3. What was written

The fits table shows how little the two numbers move from fold to fold (the ten Alpha folds share the fit on all eight Beta stations). The outcome counts show how many held-out days were corrected, kept, or sent to review.

In [ ]:
display(out['fits'][['fold_id', 'n_train', 'n_train_rpf', 'cal_intercept', 'cal_slope', 'raw_threshold_correct']].round(3))
out['site_days'].groupby(['cohort', 'outcome']).size().unstack(fill_value=0)

## Result

M9's decisions for every held-out site-day are on disk beside M7's and M8's. The next notebook scores all three against the reference.